# Mystery Dataset 1: From Data Wrangling to End-to-End Machine Learning

### Hands-On AI/ML for Biomedical Science and Engineering

This notebook takes the **mystery dataset** you have already explored and uses it to walk through many of the same ideas as an end-to-end machine-learning project. It is configured for **interactive Matplotlib figures in Google Colab**, and the same plotting approach also works in VS Code/Jupyter when `ipympl` is installed.

We will use the same dataset in **two different ways**:

1. **Supervised regression:** can we predict `Step_Height` for an animal the model has never seen before?
2. **Unsupervised discovery:** can multivariate recovery patterns help us guess which animals were treated, before the treatment identities are revealed?

---

## Learning goals

By the end, you should be able to:

- inspect a new biomedical dataset and identify its structure
- distinguish identifiers, experimental conditions, features, and outcomes
- recognize missing data and suspicious outliers
- define the **unit of independent generalization**
- make a train/test split that respects repeated measurements
- build useful derived features
- construct a preprocessing pipeline
- compare regression models with grouped cross-validation
- evaluate a final model on untouched animals
- recognize collinearity and redundant predictors
- think critically about target leakage, proxy variables, and feature availability
- reduce longitudinal biomechanics data to one row per animal
- use PCA and clustering for a blind exploratory treatment guess

---

## A Chapter 2-style workflow

A useful machine-learning workflow is:

1. Frame the problem
2. Get and inspect the data
3. Set aside a test set
4. Explore the training data
5. Prepare the data
6. Select and train models
7. Evaluate and fine-tune
8. Interpret, communicate, and sanity-check the result

The details change in biomedical science because observations are often **repeated, hierarchical, longitudinal, and correlated within subjects**.

# 0. Setup

This notebook expects `mystery_dataset_1.csv` to be either:

- in the same folder as the notebook,
- in a `data/` folder,
- or one directory above in `data/`.

In Colab, you can also simply upload the CSV into the runtime before running this cell.

## Interactive plotting setup

For this course, we will use Matplotlib in two modes:

- **interactive exploration**: zoom, pan, inspect points
- **finished output**: export a reproducible figure with `fig.savefig(...)`

This notebook targets **Google Colab** first. The next cell installs `ipympl` and `mplcursors` in Colab and enables Colab's widget manager.

In a local VS Code/Jupyter environment, install these once in the course environment:

```bash
conda install -c conda-forge ipympl mplcursors
```

Then the same `%matplotlib widget` command gives interactive figures inside the notebook.

To return to static notebook figures at any time:

```python
%matplotlib inline
```

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ipympl", "mplcursors"],
        check=True,
    )

    from google.colab import output
    output.enable_custom_widget_manager()

# Interactive Matplotlib backend in Colab, VS Code, or JupyterLab.
%matplotlib widget

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplcursors

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    cross_validate,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering

pd.set_option("display.max_columns", 100)

In [ ]:
candidate_paths = [
    Path("mystery_dataset_1.csv"),
    Path("data/mystery_dataset_1.csv"),
    Path("../data/mystery_dataset_1.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find mystery_dataset_1.csv. "
        "Put it beside this notebook, in data/, or upload it to the Colab runtime."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

df.head()

# 1. Revisit the original mystery

Before we model anything, revisit the questions you were originally given.

### Mystery questions

- What does the dataset look like it is about?
- What look like outcome variables?
- What look like identifier variables?
- Does it have missing entries?
- How many unique subjects were there?
- Was this a single time point or longitudinal study?
- Did any outcome variables change over time?
- Some animals were treated, and some were not. Can we make a blind guess about which were which?

For now, **do not worry about treatment prediction**. First understand the table.

In [ ]:
print("Rows, columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.info()

## 1A. How many unique values does each column contain?

This is often surprisingly informative:

- a column with one unique value may be a constant study label
- a column with one value per row may be an identifier
- a column with a small number of values may encode an experimental condition

In [ ]:
unique_counts = df.nunique(dropna=False).sort_values()

unique_counts

## 1B. Subjects, time points, and experimental conditions

In [ ]:
print("Unique rats:", df["Rat"].nunique())
print("Rat IDs:", sorted(df["Rat"].unique()))

print("\nTime points:")
print(df["Timepoint_Name"].value_counts())

print("\nNominal speed groups:")
print(df["Speed_Group"].value_counts())

print("\nStudy labels:")
print(df["Study"].value_counts())

### Pause and classify the columns

A useful *working* classification is:

- **Identifiers:** identify a row, file, trial, stride, or animal
- **Experimental conditions:** things imposed or known about the observation
- **Outcome measurements:** measured gait/kinematic quantities
- **Target:** the one outcome we choose to predict

There can be debate here. That's normal. The classification depends partly on the scientific question.

### Your turn

Which columns would you absolutely **not** give to a model if the goal were to generalize to a new animal?

In [ ]:
# A working list to discuss in class.
# We will NOT use these as model features.

identifier_like = [
    "Unnamed: 0",
    "Name_File",
    "Study",
    "Rat",
    "Trial_Name",
    "Stride",
]

df[identifier_like].head()

# 2. Missing data

First ask **where** data are missing, not merely whether any `NaN` exists.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)

missing[missing > 0]

If several biomechanical variables disappear on exactly the same rows, that may represent a **failed measurement or failed processing step**, not independent random missing values.

That distinction matters scientifically.

### Question

Would replacing every missing kinematic value with the column median necessarily be the scientifically correct thing to do?

We'll use median imputation later because it is a useful machine-learning pipeline example, but preprocessing decisions should ultimately be justified by how the data were generated.

In [ ]:
missing_rows = df[df.isna().any(axis=1)]

print("Rows with at least one missing value:", len(missing_rows))

missing_trials = (
    missing_rows[
        ["Rat", "Timepoint_Name", "Speed_Group", "Trial_Name"]
    ]
    .drop_duplicates()
    .sort_values(["Rat", "Timepoint_Name", "Speed_Group"])
)

missing_trials

# 3. Is this longitudinal?

A row is a stride, but the experiment is organized at several levels:

**strides → trials → speeds → time points → animals**

Let's inspect how many observations each animal contributes at each time point.

In [ ]:
rat_time_counts = pd.crosstab(
    df["Rat"],
    df["Timepoint_Name"],
)

rat_time_counts

## A quick longitudinal view

To avoid mixing every speed together, let's look at one nominal speed group and summarize each rat/time point by its **median** `Step_Height`.

This is exploratory visualization, not yet a formal statistical analysis.

In [ ]:
time_order = [
    "week-1", "week1", "week2", "week4",
    "week6", "week7", "week8", "week9"
]

example_speed = "speed24"

trajectory = (
    df[df["Speed_Group"] == example_speed]
    .groupby(["Rat", "Timepoint_Name"])["Step_Height"]
    .median()
    .unstack()
    .reindex(columns=time_order)
)

ax = trajectory.T.plot(
    marker="o",
    figsize=(10, 6),
    legend=False,
)

ax.set_xlabel("Time point")
ax.set_ylabel("Median Step Height")
ax.set_title(f"Step Height trajectories at {example_speed}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interactive figure controls

With `%matplotlib widget`, use the figure toolbar to pan and zoom.

For a reproducible export, save explicitly from code:

```python
fig.savefig("step_height_trajectories.png", dpi=300, bbox_inches="tight")
```

The toolbar is great for exploration. `savefig()` is preferable when the exact output matters.

# 4. Scout candidate outcomes before choosing a target

A target should not be chosen only because it gives the largest correlation or the easiest model.

Before settling on `Step_Height`, let's ask two descriptive questions:

1. Which candidate outcomes vary strongly with **belt speed**?
2. Which candidate outcomes show strong **baseline → week 4** longitudinal change?

This is dataset orientation, not model tuning. Once we define the prediction problem and create a test set, later EDA/model choices should use the training animals only.

In [ ]:
candidate_outcomes = [
    "Step_Height",
    "Step_Length",
    "Step_Duration",
    "Stance_Duration",
    "Mean_Asis_Toe_Height",
    "Mean_Ankle",
    "Mean_Ankle_Swing",
    "Variation_Ankle_Swing",
    "Mean_Knee",
    "Variation_Knee_Swing",
    "Mean_Hip",
    "Variation_Hip_Swing",
]

speed_correlations = (
    df[candidate_outcomes + ["Belt_Speed"]]
    .corr(numeric_only=True)["Belt_Speed"]
    .drop("Belt_Speed")
    .sort_values(key=np.abs, ascending=False)
)

speed_correlations.to_frame("Pearson_r_with_Belt_Speed")

The timing variables are expected to be strongly related to treadmill speed. That can make them scientifically sensible targets, but also somewhat **too easy** for our first regression exercise.

`Step_Height` is the maximum paw clearance during swing. It is biologically intuitive, continuous, and meaningfully predictable, while not being dominated by belt speed alone. That makes it a useful teaching target.

In [ ]:
# A quick rat-level longitudinal scout.
# First summarize strides within speed, then summarize across speeds.

scout_rat_time_speed = (
    df
    .groupby(["Rat", "Timepoint_Name", "Speed_Group"])[candidate_outcomes]
    .median()
    .reset_index()
)

scout_rat_time = (
    scout_rat_time_speed
    .groupby(["Rat", "Timepoint_Name"])[candidate_outcomes]
    .median()
)

baseline_scout = scout_rat_time.xs("week-1", level="Timepoint_Name")
week4_scout = scout_rat_time.xs("week4", level="Timepoint_Name")

paired = baseline_scout.join(
    week4_scout,
    lsuffix="_baseline",
    rsuffix="_week4",
    how="inner",
)

effect_rows = []

for outcome in candidate_outcomes:
    delta = (
        paired[f"{outcome}_week4"]
        - paired[f"{outcome}_baseline"]
    ).dropna()

    sd_delta = delta.std(ddof=1)
    dz = delta.mean() / sd_delta if sd_delta > 0 else np.nan

    effect_rows.append({
        "Outcome": outcome,
        "Mean_change_week4_minus_baseline": delta.mean(),
        "Paired_effect_size_dz": dz,
        "N_rats": len(delta),
    })

longitudinal_scout = (
    pd.DataFrame(effect_rows)
    .sort_values(
        "Paired_effect_size_dz",
        key=np.abs,
        ascending=False,
    )
)

longitudinal_scout

### Why keep `Step_Height`?

Some joint-angle and ROM-like variables show stronger longitudinal changes, and timing variables are more obviously speed dependent.

We're keeping `Step_Height` because it creates a richer modeling exercise:

- it is easy to interpret biologically
- it is not trivially determined by treadmill speed
- it has useful geometric/kinematic predictors
- it creates a natural later discussion about correlated predictors and related measures such as ASIS-toe height
- it remains relevant when we return to recovery and treatment effects

So the target is chosen for **scientific interpretability + teaching value**, not because it wins a univariate screening contest.

# 5. Frame a supervised ML problem

For the Chapter-2-style part of the exercise, let's define a concrete prediction problem:

> **Given other gait and joint-kinematic measurements from a stride, predict `Step_Height` for an animal that was not used to train the model.**

This is:

- **supervised learning**
- **regression**
- a prediction problem, **not** a causal analysis

The target is:

```python
Step_Height
```

### Critical question

What should the test set represent?

If the intended claim is:

> "This should work on a new animal"

then the test set must contain **whole animals the model has never seen**.

# 6. The most important biomedical wrinkle: repeated observations

There are thousands of rows, but far fewer independent animals.

A naive random row split can place:

- stride 12 from Rat 162 into training
- stride 19 from Rat 162 into training
- stride 27 from Rat 162 into testing

That makes the test set much less independent than it looks.

Let's demonstrate.

In [ ]:
from sklearn.model_selection import train_test_split

naive_train, naive_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
)

rats_in_both = set(naive_train["Rat"]) & set(naive_test["Rat"])

print("Rats represented in BOTH naive train and test:", len(rats_in_both))
print(sorted(rats_in_both))

## Split by animal instead

`GroupShuffleSplit` lets us keep all strides from a rat together.

In [ ]:
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    group_split.split(df, groups=df["Rat"])
)

train_raw = df.iloc[train_idx].copy()
test_raw = df.iloc[test_idx].copy()

print("Training rats:", sorted(train_raw["Rat"].unique()))
print("Test rats:    ", sorted(test_raw["Rat"].unique()))

overlap = set(train_raw["Rat"]) & set(test_raw["Rat"])

print("\nRat overlap:", overlap)
print("Training rows:", len(train_raw))
print("Test rows:", len(test_raw))

From this point forward:

## 🔒 Pretend `test_raw` is locked in a drawer.

Do exploration, preprocessing decisions, model comparison, and tuning using **training data only**.

We will open the test set once at the end.

# 7. Explore the training data

## 7A. Descriptive statistics

In [ ]:
train_raw[
    [
        "Step_Height",
        "Step_Length",
        "Step_Duration",
        "Stance_Duration",
        "Belt_Speed",
    ]
].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99, 0.995]
)

## 7B. Look for suspicious extremes

Biomedical datasets often contain values that are:

- biologically extreme but real
- tracking errors
- unit problems
- processing failures
- transcription errors

Do **not** automatically delete outliers merely because they are inconvenient.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(train_raw["Step_Height"].dropna(), bins=80)
axes[0].set_xlabel("Step Height")
axes[0].set_ylabel("Count")
axes[0].set_title("Full Step Height distribution")

upper = train_raw["Step_Height"].quantile(0.995)

axes[1].hist(
    train_raw.loc[train_raw["Step_Height"] <= upper, "Step_Height"],
    bins=60,
)
axes[1].set_xlabel("Step Height")
axes[1].set_ylabel("Count")
axes[1].set_title("Zoomed to 99.5th percentile")

plt.tight_layout()
plt.show()

print("Largest Step Height values:")
display(
    train_raw.nlargest(10, "Step_Height")[
        ["Rat", "Timepoint_Name", "Speed_Group",
         "Step_Height", "Step_Length", "Trial_Name"]
    ]
)

### Discussion

Before removing an extreme observation, ask:

1. Is the value physically plausible?
2. Does the raw recording support it?
3. Is there a documented tracking failure?
4. Would the same rule be applied to future data?
5. Was the rule defined using training data only?

For this first ML exercise, we'll leave the observations in place and let model errors reveal some of the consequences.

## 7C. Correlations with Step Height

Correlation is not causation, and correlated predictors are common in biomechanics.

Still, this is a useful first look.

In [ ]:
exclude_for_correlation = [
    "Unnamed: 0",
    "Rat",
    "Stride",
]

numeric_training = (
    train_raw
    .drop(columns=exclude_for_correlation, errors="ignore")
    .select_dtypes(include="number")
)

correlations = (
    numeric_training
    .corr()["Step_Height"]
    .drop("Step_Height")
    .sort_values(key=np.abs, ascending=False)
)

correlations.head(15)

# 8. Correlated and redundant predictors

Biomedical variables often describe overlapping aspects of the same movement.

Examples in this dataset include:

- `Belt_Speed` and `Speed_Group`
- `Step_Duration` and `Stance_Duration`
- step geometry and related toe-height measures
- joint summaries calculated from the same stride

This is not automatically a problem, but it changes how we interpret models.

## Collinearity

When two predictors carry nearly the same information:

- a linear model can distribute the effect between them in unstable or unintuitive ways
- individual coefficients can change substantially when one correlated predictor is added or removed
- prediction can still be good even when coefficient interpretation is poor

Tree models handle redundancy differently, but feature importance can also be divided among correlated variables.

In [ ]:
correlation_demo_cols = [
    "Belt_Speed",
    "Step_Length",
    "Step_Duration",
    "Stance_Duration",
    "Step_Height",
    "Mean_Asis_Toe_Height",
    "Mean_Ankle",
    "Mean_Knee",
    "Mean_Hip",
]

corr_demo = (
    train_raw[correlation_demo_cols]
    .corr(numeric_only=True)
    .round(2)
)

corr_demo

## A deliberately redundant representation of speed

`Speed_Group` is essentially a categorical representation of the same experimental quantity described continuously by `Belt_Speed`.

Giving a linear model **both** can make its coefficients awkward to interpret.

Let's compare:

- Model A: continuous `Belt_Speed`
- Model B: `Belt_Speed` **plus** categorical `Speed_Group`

The point is not which model gets a better score. Watch what happens to the coefficient assigned to `Belt_Speed`.

In [ ]:
demo_numeric = [
    "Belt_Speed",
    "Step_Length",
    "Step_Duration",
    "Stance_Duration",
]

# Model A: continuous speed only
prep_a = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        demo_numeric,
    )
])

lin_a = Pipeline([
    ("prep", prep_a),
    ("model", LinearRegression()),
])

lin_a.fit(
    train_raw[demo_numeric],
    train_raw["Step_Height"],
)

coef_a = pd.Series(
    lin_a.named_steps["model"].coef_,
    index=lin_a.named_steps["prep"].get_feature_names_out(),
)

# Model B: same numeric predictors PLUS categorical Speed_Group
prep_b = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        demo_numeric,
    ),
    (
        "speed_group",
        OneHotEncoder(handle_unknown="ignore"),
        ["Speed_Group"],
    ),
])

lin_b = Pipeline([
    ("prep", prep_b),
    ("model", LinearRegression()),
])

lin_b.fit(
    train_raw[demo_numeric + ["Speed_Group"]],
    train_raw["Step_Height"],
)

coef_b = pd.Series(
    lin_b.named_steps["model"].coef_,
    index=lin_b.named_steps["prep"].get_feature_names_out(),
)

print("Model A: Belt_Speed coefficient")
display(coef_a[coef_a.index.str.contains("Belt_Speed")])

print("\nModel B: Belt_Speed + Speed_Group coefficients")
display(coef_b[coef_b.index.str.contains("Belt_Speed|speed_group")])

### Interpretation

If the `Belt_Speed` coefficient changes markedly after adding `Speed_Group`, nothing biological suddenly changed.

The model is being asked to divide the same information among redundant predictors.

This is one reason to distinguish:

> **good prediction**

from

> **clean interpretation of individual coefficients**

For our main models below, we will use **continuous `Belt_Speed`** and leave out `Speed_Group` so we do not feed the same speed information twice.

# 9. Feature engineering

Good features often encode scientifically meaningful relationships.

Two simple gait-derived quantities are:

### Swing duration

```text
Swing Duration = Step Duration - Stance Duration
```

### Duty factor

```text
Duty Factor = Stance Duration / Step Duration
```

These are constructed only from predictor information, not from `Step_Height`.

In [ ]:
def add_derived_features(frame):
    out = frame.copy()

    out["Swing_Duration"] = (
        out["Step_Duration"] - out["Stance_Duration"]
    )

    out["Duty_Factor"] = (
        out["Stance_Duration"] / out["Step_Duration"]
    )

    return out


train = add_derived_features(train_raw)
test = add_derived_features(test_raw)

train[
    [
        "Step_Duration",
        "Stance_Duration",
        "Swing_Duration",
        "Duty_Factor",
    ]
].head()

# 10. Choose predictors

For our **first model**, we'll deliberately leave out the ASIS-toe-height variables.

Why?

They may contain information very closely related to the geometry underlying `Step_Height`. Whether that is appropriate depends on the intended use of the model.

Later, we'll deliberately add an ASIS-height variable and ask:

> Is this a legitimate predictor, a redundant proxy, or information that would not really be available when we want to make the prediction?

That is a more useful question than automatically declaring every highly predictive variable "leakage."

We also deliberately use **continuous `Belt_Speed` without `Speed_Group`** in the main model. They encode essentially the same experimental quantity, and using both would make coefficient interpretation unnecessarily muddy.


In [ ]:
target = "Step_Height"

joint_features = [
    col for col in train.columns
    if (
        col.startswith("Mean_Ankle")
        or col.startswith("Variation_Ankle")
        or col.startswith("Mean_Knee")
        or col.startswith("Variation_Knee")
        or col.startswith("Mean_Hip")
        or col.startswith("Variation_Hip")
    )
]

numeric_features = [
    "Belt_Speed",
    "Step_Length",
    "Step_Duration",
    "Stance_Duration",
    "Swing_Duration",
    "Duty_Factor",
] + joint_features

categorical_features = [
    "Timepoint_Name",
]

feature_columns = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)
print("Total model input columns:", len(feature_columns))

In [ ]:
X_train = train[feature_columns]
y_train = train[target]

X_test = test[feature_columns]
y_test = test[target]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# 11. Build a preprocessing pipeline

Our table contains:

- numerical variables
- categorical variables
- some missing kinematic values

For numerical variables:

1. replace missing values with the training-set median
2. standardize to approximately zero mean / unit variance

For categorical variables:

1. replace missing categories if needed
2. one-hot encode the categories

Putting these steps into a `Pipeline`/`ColumnTransformer` helps prevent accidental preprocessing leakage.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

preprocessor

# 12. Establish a boring baseline

Before celebrating a fancy model, ask whether it beats a trivial prediction.

Here the dummy model predicts the **median training Step Height** for everything.

In [ ]:
dummy_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", DummyRegressor(strategy="median")),
])

# 13. Compare several model families

We'll try:

- Dummy regression
- Linear regression
- A decision tree
- A random forest

But we still should **not use the held-out test rats to choose among them**.

Instead, perform cross-validation **inside the training animals**.

In [ ]:
models = {
    "Dummy": DummyRegressor(strategy="median"),

    "Linear regression": LinearRegression(),

    "Decision tree": DecisionTreeRegressor(
        max_depth=8,
        random_state=42,
    ),

    "Random forest": RandomForestRegressor(
        n_estimators=10,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
}

# Four folds keeps whole rats together while staying quick in class.
group_cv = GroupKFold(n_splits=3)

rows = []

for model_name, estimator in models.items():

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator),
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        groups=train["Rat"],
        cv=group_cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2",
        },
        n_jobs=1,
    )

    rows.append({
        "Model": model_name,
        "MAE": -scores["test_MAE"].mean(),
        "RMSE": -scores["test_RMSE"].mean(),
        "R2": scores["test_R2"].mean(),
        "R2_SD": scores["test_R2"].std(),
    })

cv_results = (
    pd.DataFrame(rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

cv_results

### Questions

- Which model has the lowest cross-validated RMSE?
- Does the best model win on every metric?
- How variable is performance across groups of held-out animals?
- Why might animal-to-animal variability matter more than stride count here?

# 14. Optional stretch: small hyperparameter search

Chapter-2-style workflows often tune hyperparameters using cross-validation.

This can take longer, so it is left as an optional exercise.

**Important:** the folds still need to respect rat identity.

Uncomment the cell if you want to run it.

In [ ]:
# OPTIONAL STRETCH
#
# from sklearn.model_selection import GridSearchCV
#
# forest_pipeline = Pipeline([
#     ("preprocess", preprocessor),
#     ("model", RandomForestRegressor(
#         random_state=42,
#         n_jobs=-1,
#     )),
# ])
#
# param_grid = {
#     "model__n_estimators": [50, 100],
#     "model__max_depth": [None, 12],
#     "model__min_samples_leaf": [1, 3],
# }
#
# search = GridSearchCV(
#     forest_pipeline,
#     param_grid=param_grid,
#     cv=GroupKFold(n_splits=3),
#     scoring="neg_root_mean_squared_error",
#     n_jobs=1,
# )
#
# search.fit(
#     X_train,
#     y_train,
#     groups=train["Rat"],
# )
#
# print("Best parameters:")
# print(search.best_params_)
#
# print("\nBest grouped-CV RMSE:")
# print(-search.best_score_)

# 15. Final evaluation on untouched rats

Now choose a model based on training-set cross-validation.

For this teaching example we'll use the random forest.

Fit it once on **all training rats**, then evaluate once on the held-out test rats.

In [ ]:
final_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=10,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    )),
])

final_model.fit(X_train, y_train)

pred = final_model.predict(X_test)

test_mae = mean_absolute_error(y_test, pred)
test_rmse = mean_squared_error(y_test, pred) ** 0.5
test_r2 = r2_score(y_test, pred)

print(f"Held-out rats: {sorted(test['Rat'].unique())}")
print(f"MAE:  {test_mae:.3f}")
print(f"RMSE: {test_rmse:.3f}")
print(f"R²:   {test_r2:.3f}")

### Don't tune against this result repeatedly

Once you inspect the test performance, it is no longer a completely untouched source of evidence.

Repeatedly changing the model until the test score looks good turns the test set into another validation set.

## 15A. Predicted versus observed

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(y_test, pred, alpha=0.35)

cursor = mplcursors.cursor(sc, hover=True)
@cursor.connect("add")
def _hover_prediction(sel):
    row_pos = sel.index
    rat = test.iloc[row_pos]["Rat"]
    sel.annotation.set_text(
        f"Rat {rat}\nObserved = {sel.target[0]:.2f}\nPredicted = {sel.target[1]:.2f}"
    )

low = min(y_test.min(), pred.min())
high = max(y_test.max(), pred.max())

ax.plot([low, high], [low, high], linestyle="--")

ax.set_xlabel("Observed Step Height")
ax.set_ylabel("Predicted Step Height")
ax.set_title("Held-out animals: predicted vs observed")
plt.tight_layout()
plt.show()

## 15B. Residuals

Residual:

```text
observed - predicted
```

A useful model should not have an obvious systematic residual pattern.

In [ ]:
residuals = y_test.to_numpy() - pred

fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(pred, residuals, alpha=0.35)
ax.axhline(0, linestyle="--")

ax.set_xlabel("Predicted Step Height")
ax.set_ylabel("Residual")
ax.set_title("Residuals on held-out animals")

plt.tight_layout()
plt.show()

# 16. What did the random forest use?

Tree-based models can provide a rough measure of feature importance.

This does **not** prove causation or biological importance. It tells us which inputs the fitted model found useful for prediction.

In [ ]:
feature_names = (
    final_model
    .named_steps["preprocess"]
    .get_feature_names_out()
)

importances = (
    final_model
    .named_steps["model"]
    .feature_importances_
)

importance_table = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances,
    })
    .sort_values("importance", ascending=False)
)

importance_table.head(15)

In [ ]:
top = importance_table.head(15).sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 6))

ax.barh(top["feature"], top["importance"])
ax.set_xlabel("Random-forest feature importance")
ax.set_title("Top predictive features")

plt.tight_layout()
plt.show()

# 17. ASIS-height experiment: leakage, proxy, or legitimate feature?

Now deliberately add:

```python
Mean_Asis_Toe_Height
```

and ask whether model performance changes.

### Important distinction

A feature is not automatically "leakage" merely because it is highly predictive.

Ask instead:

- Was this measurement available at the time the prediction would be made?
- Was it calculated from the target itself?
- Is it essentially another measurement of the same physical quantity?
- Would we have this feature in the intended real-world application?

In biomechanics, simultaneous geometrically related measures can be **redundant proxies** without necessarily being classical train/test leakage.

In [ ]:
asis_feature = "Mean_Asis_Toe_Height"

numeric_features_with_asis = numeric_features + [asis_feature]
feature_columns_with_asis = (
    numeric_features_with_asis + categorical_features
)

numeric_pipeline_with_asis = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_with_asis = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline_with_asis,
        numeric_features_with_asis,
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features,
    ),
])

forest_with_asis = Pipeline([
    ("preprocess", preprocessor_with_asis),
    ("model", RandomForestRegressor(
        n_estimators=10,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    )),
])

asis_scores = cross_validate(
    forest_with_asis,
    train[feature_columns_with_asis],
    y_train,
    groups=train["Rat"],
    cv=GroupKFold(n_splits=3),
    scoring={
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2",
    },
    n_jobs=1,
)

print(
    "With ASIS mean height, grouped-CV RMSE:",
    round(-asis_scores["test_RMSE"].mean(), 3)
)
print(
    "With ASIS mean height, grouped-CV R²:",
    round(asis_scores["test_R2"].mean(), 3)
)

print("\nCompare with the original Random Forest row:")
display(
    cv_results[
        cv_results["Model"] == "Random forest"
    ]
)

# 18. Return to the original mystery: which animals were treated?

Now we switch problems.

The treatment identities are hidden, so we **do not have labels** for supervised learning.

That means:

> "Predict treated vs untreated"

is not currently a supervised classification problem.

Instead, we can ask whether animals show **different multivariate recovery patterns**.

That is an **unsupervised exploratory problem**.

# 19. Change the unit of analysis before PCA

This is crucial.

Treatment was applied to **animals**, not individual strides.

So we should not feed 16,000 correlated strides into PCA and pretend we have 16,000 independent treatment examples.

Instead:

1. summarize strides within each rat × time point × speed
2. give each speed roughly equal weight
3. calculate change from baseline (`week-1`) to a common post-injury time (`week4`)
4. make **one row per rat**

In [ ]:
mystery_outcomes = [
    "Step_Height",
    "Step_Length",
    "Stance_Duration",
    "Mean_Asis_Toe_Height",
    "Mean_Ankle",
    "Mean_Knee",
    "Mean_Hip",
]

# First summarize strides within each nominal speed.
rat_time_speed = (
    df
    .groupby(
        ["Rat", "Timepoint_Name", "Speed_Group"]
    )[mystery_outcomes]
    .median()
    .reset_index()
)

# Then take the median across nominal speeds so that a speed
# with more recorded strides does not automatically dominate.
rat_time = (
    rat_time_speed
    .groupby(["Rat", "Timepoint_Name"])[mystery_outcomes]
    .median()
)

baseline = (
    rat_time
    .xs("week-1", level="Timepoint_Name")
)

week4 = (
    rat_time
    .xs("week4", level="Timepoint_Name")
)

change_week4 = (
    week4 - baseline
).dropna()

print("One row per rat:", change_week4.shape)

change_week4

### Why week 4?

For this first blind exercise, `week4` is useful because it is a common post-baseline time point represented across the animals.

Later, a better longitudinal model could use the **entire recovery trajectory** rather than collapsing it to one change score.

# 20. Standardize before PCA

The variables have different units and scales.

Without standardization, a numerically large variable could dominate the PCA simply because of its units.

In [ ]:
scaler = StandardScaler()

Z = scaler.fit_transform(change_week4)

pca = PCA(n_components=2)

PC = pca.fit_transform(Z)

pca_table = pd.DataFrame(
    PC,
    index=change_week4.index,
    columns=["PC1", "PC2"],
)

print(
    "Explained variance ratio:",
    np.round(pca.explained_variance_ratio_, 3)
)

pca_table

## Plot the animals in PCA space

The axes are new combinations of the original recovery measures.

Do any animals appear to separate into groups?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    pca_table["PC1"],
    pca_table["PC2"],
    s=70,
)

cursor = mplcursors.cursor(sc, hover=True)

@cursor.connect("add")
def _hover_pca(sel):
    rat = pca_table.index[sel.index]
    sel.annotation.set_text(
        f"Rat {rat}\nPC1 = {sel.target[0]:.2f}\nPC2 = {sel.target[1]:.2f}"
    )

for rat, row in pca_table.iterrows():
    ax.annotate(
        str(rat),
        (row["PC1"], row["PC2"]),
        xytext=(5, 4),
        textcoords="offset points",
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Blind recovery patterns: baseline → week 4")

plt.tight_layout()
plt.show()

# 21. What do the principal components represent?

PCA gives us components, but interpretation still requires going back to the original variables.

The **loadings** show how strongly each original standardized variable contributes to each PC.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=mystery_outcomes,
    columns=["PC1_loading", "PC2_loading"],
)

loadings.sort_values(
    "PC1_loading",
    key=np.abs,
    ascending=False,
)

# 22. Blind clustering

Let's ask two clustering algorithms to split the 15 animals into two groups.

The cluster labels `0` and `1` are arbitrary. Neither automatically means "treated."

In [ ]:
kmeans = KMeans(
    n_clusters=2,
    n_init=20,
    random_state=42,
)

kmeans_cluster = kmeans.fit_predict(Z)

hierarchical = AgglomerativeClustering(
    n_clusters=2
)

hier_cluster = hierarchical.fit_predict(Z)

blind_guess = pca_table.copy()

blind_guess["KMeans_cluster"] = kmeans_cluster
blind_guess["Hierarchical_cluster"] = hier_cluster

blind_guess = (
    blind_guess
    .reset_index()
    .sort_values(["KMeans_cluster", "Rat"])
)

blind_guess

### Do the clustering methods agree?

Agreement does not prove the clusters are biologically real.

Disagreement is also informative: it tells us the grouping is not robust to how similarity is defined.

In [ ]:
pd.crosstab(
    blind_guess["KMeans_cluster"],
    blind_guess["Hierarchical_cluster"],
    rownames=["KMeans"],
    colnames=["Hierarchical"],
)

## Visualize the K-means assignment

In [ ]:
plot_df = blind_guess.set_index("Rat")

fig, ax = plt.subplots(figsize=(8, 6))

for cluster in sorted(plot_df["KMeans_cluster"].unique()):

    subset = plot_df[
        plot_df["KMeans_cluster"] == cluster
    ]

    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        s=80,
        label=f"Cluster {cluster}",
    )

    for rat, row in subset.iterrows():
        ax.annotate(
            str(rat),
            (row["PC1"], row["PC2"]),
            xytext=(5, 4),
            textcoords="offset points",
        )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Blind two-cluster solution")
ax.legend()

plt.tight_layout()
plt.show()

# 23. Translate the clustering back to biology

A cluster number by itself means nothing.

Let's compare the average **baseline → week 4 change** of the original variables in each K-means group.

In [ ]:
change_with_cluster = change_week4.copy()

change_with_cluster["Cluster"] = (
    pd.Series(
        kmeans_cluster,
        index=change_week4.index,
    )
)

cluster_change_summary = (
    change_with_cluster
    .groupby("Cluster")
    .mean()
    .T
)

cluster_change_summary

### Questions before making a blind guess

- Which outcome changes distinguish the clusters?
- Does one cluster look more like "recovery"?
- Are the apparent differences dominated by one or two unusual rats?
- Do PCA and clustering tell the same story?
- Does your scientific interpretation change if you remove one feature?
- Does your answer survive changing from median to mean aggregation?

Record your **blind treatment guess before the identities are revealed**.

In [ ]:
# Export a simple blind-guess table if desired.

blind_guess.to_csv(
    "mystery_dataset_blind_cluster_guess.csv",
    index=False,
)

blind_guess

# 24. Why not dPCA yet?

**Demixed PCA (dPCA)** can be useful when we already know experimental factors and want to separate variation associated with them, for example:

- treadmill speed
- time after injury
- treatment group
- interactions among those factors

But at this point **treatment is deliberately hidden**.

So ordinary PCA is the cleaner first tool for the blind discovery problem.

After treatment identities are revealed, it could be interesting later in the course to revisit the full longitudinal dataset and ask whether variation can be separated into dimensions associated with:

```text
speed × time × treatment
```

That would be a different question from today's blind clustering exercise.

# 25. Take-home lessons

### Data science lessons

- Inspect structure before modeling.
- A large row count does not guarantee a large number of independent subjects.
- Missingness has a mechanism.
- Outliers deserve scientific investigation.
- Feature engineering should encode meaningful relationships.
- Correlated predictors can support good prediction while making individual coefficients hard to interpret.
- Preprocessing belongs inside reproducible pipelines.
- Establish a boring baseline before celebrating a model.
- Use cross-validation for model selection.
- Keep the final test set genuinely held out.

### Biomedical ML lessons

- Split according to the scientific unit you want to generalize to.
- Repeated strides from the same rat are correlated.
- Prediction is not causation.
- Highly predictive features can be useful, redundant, or inappropriate depending on when and how they are measured.
- Treatment-level analyses should ultimately operate at the treatment unit: the animal.
- PCA finds directions of variation; it does not discover biological truth.
- Clusters are hypotheses to investigate, not labels handed down by mathematics.

### The mystery continues

After the treatment labels are revealed, we can finally turn the blind treatment question into a **supervised** problem and ask whether the same features genuinely discriminate treatment groups in held-out animals.